# Projet ML Retail - Analyse Comportementale Clientele

## Module Machine Learning - GI2

Ce notebook presente l'exploration complete des donnees et le prototypage des modeles.

## 1. Configuration et Imports

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Imports OK")

## 2. Chargement des Donnees

In [ ]:
# Charger les donnees
df = pd.read_csv('../data/raw/retail_customers_COMPLETE_CATEGORICAL.csv')
print(f"Shape: {df.shape}")
print(f"Colonnes: {df.columns.tolist()}")

In [ ]:
# Apercu des donnees
df.head()

In [ ]:
# Info sur les types
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

## 3. Exploration des Donnees

### 3.1 Valeurs Manquantes

In [ ]:
# Analyse des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Manquantes': missing,
    'Pourcentage': missing_pct
})

missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)

In [ ]:
# Visualisation des valeurs manquantes
plt.figure(figsize=(14, 6))
missing_cols = missing[missing > 0].sort_values(ascending=False)
missing_pct_plot = (missing_cols / len(df) * 100)

plt.bar(range(len(missing_cols)), missing_pct_plot.values, color='coral')
plt.xticks(range(len(missing_cols)), missing_cols.index, rotation=45, ha='right')
plt.xlabel('Colonnes')
plt.ylabel('Pourcentage manquant (%)')
plt.title('Pourcentage de valeurs manquantes par colonne')
plt.tight_layout()
plt.show()

### 3.2 Distribution de la Variable Cible (Churn)

In [ ]:
# Distribution du Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Comptes
churn_counts = df['Churn'].value_counts()
axes[0].bar(['Fidele (0)', 'Parti (1)'], churn_counts.values, color=['steelblue', 'coral'])
axes[0].set_ylabel('Nombre de clients')
axes[0].set_title('Distribution du Churn')

# Pourcentages
churn_pct = df['Churn'].value_counts(normalize=True) * 100
axes[1].pie(churn_pct.values, labels=['Fidele (0)', 'Parti (1)'], 
            autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title('Proportion du Churn')

plt.tight_layout()
plt.show()

print(f"\nDesequilibre des classes:")
print(churn_counts)

### 3.3 Distribution des Features Numeriques

In [ ]:
# Selectionner les colonnes numeriques principales
numeric_features = ['Recency', 'Frequency', 'MonetaryTotal', 'MonetaryAvg', 
                    'TotalQuantity', 'CustomerTenureDays', 'Age']

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if col in df.columns:
        df[col].hist(bins=50, ax=axes[i], color='steelblue', edgecolor='black')
        axes[i].set_title(f'Distribution de {col}')
        axes[i].set_xlabel(col)

# Supprimer les axes vides
for j in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### 3.4 Matrice de Correlation

In [ ]:
# Matrice de correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1)
plt.title('Matrice de Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation avec Churn
churn_corr = corr_matrix['Churn'].sort_values(ascending=False)
print("Correlation avec Churn:")
print(churn_corr)

### 3.5 Features Categorielles

In [ ]:
# Distribution des features categorielles
categorical_features = ['RFMSegment', 'SpendingCategory', 'CustomerType', 'Region', 'ChurnRiskCategory']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    if col in df.columns:
        df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue')
        axes[i].set_title(f'Distribution de {col}')
        axes[i].tick_params(axis='x', rotation=45)

# Supprimer les axes vides
for j in range(len(categorical_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Copie pour preprocessing
df_clean = df.copy()

# Supprimer les colonnes inutiles
cols_to_drop = ['CustomerID', 'NewsletterSubscribed', 'LastLoginIP', 'RegistrationDate']
df_clean = df_clean.drop([c for c in cols_to_drop if c in df_clean.columns], axis=1)

print(f"Shape apres suppression: {df_clean.shape}")

In [ ]:
# Traitement des valeurs speciales dans SupportTickets et Satisfaction
# -1 et 999 sont des valeurs speciales
df_clean.loc[df_clean['SupportTicketsCount'].isin([-1, 999]), 'SupportTicketsCount'] = np.nan
df_clean.loc[df_clean['SatisfactionScore'].isin([-1, 99]), 'SatisfactionScore'] = np.nan

print("Valeurs speciales traitees")

In [ ]:
# Imputation des valeurs manquantes numeriques par la mediane
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

# Imputation des valeurs categorielles par le mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"Valeurs manquantes restantes: {df_clean.isnull().sum().sum()}")

In [ ]:
# Encodage des variables ordinales
ordinal_mappings = {
    'SpendingCategory': {'Low': 1, 'Medium': 2, 'High': 3, 'VIP': 4},
    'ChurnRiskCategory': {'Faible': 1, 'Moyen': 2, 'Élevé': 3, 'Critique': 4, 'Eleve': 3},
    'BasketSizeCategory': {'Petit': 1, 'Moyen': 2, 'Grand': 3, 'Inconnu': 0}
}

for col, mapping in ordinal_mappings.items():
    if col in df_clean.columns:
        df_clean[col + '_encoded'] = df_clean[col].map(mapping).fillna(0).astype(int)
        print(f"Encoded: {col}")

In [ ]:
# One-Hot Encoding pour les variables nominales
nominal_cols = ['CustomerType', 'FavoriteSeason', 'Region', 'Gender', 'AccountStatus']

for col in nominal_cols:
    if col in df_clean.columns:
        dummies = pd.get_dummies(df_clean[col], prefix=col, drop_first=True)
        df_clean = pd.concat([df_clean, dummies], axis=1)
        print(f"One-Hot: {col} -> {len(dummies.columns)} colonnes")

In [ ]:
# Supprimer les colonnes categorielles originales
cols_to_drop_cat = ['RFMSegment', 'AgeCategory', 'SpendingCategory', 'CustomerType',
                    'FavoriteSeason', 'PreferredTimeOfDay', 'Region', 'LoyaltyLevel',
                    'ChurnRiskCategory', 'WeekendPreference', 'BasketSizeCategory',
                    'ProductDiversity', 'Gender', 'AccountStatus', 'Country']

df_clean = df_clean.drop([c for c in cols_to_drop_cat if c in df_clean.columns], axis=1)
print(f"Shape finale: {df_clean.shape}")

## 5. Analyse en Composantes Principales (ACP)

In [ ]:
# Preparation pour ACP
y = df_clean['Churn']
X = df_clean.drop('Churn', axis=1)

# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {X.shape[1]}")

In [ ]:
# ACP
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Variance expliquee
variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

# Trouver le nombre de composantes pour 95% de variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Nombre de composantes pour 95% de variance: {n_components_95}")

In [ ]:
# Visualisation de la variance expliquee
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance individuelle
axes[0].bar(range(1, len(variance_ratio[:20])+1), variance_ratio[:20], alpha=0.7)
axes[0].set_xlabel('Composante Principale')
axes[0].set_ylabel('Variance Expliquee')
axes[0].set_title('Variance expliquee par composante (top 20)')

# Variance cumulee
axes[1].plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'b-o', markersize=3)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].axvline(x=n_components_95, color='g', linestyle='--', label=f'n={n_components_95}')
axes[1].set_xlabel('Nombre de composantes')
axes[1].set_ylabel('Variance cumulee')
axes[1].set_title('Variance cumulee')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation 2D avec ACP
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='RdYlBu', alpha=0.6)
plt.colorbar(scatter, label='Churn')
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.title('Projection ACP 2D - Colore par Churn')
plt.show()

## 6. Clustering (K-Means)

In [ ]:
# Methode du coude et silhouette
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouettes[-1]:.4f}")

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Methode du coude
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Methode du Coude')

# Silhouette
axes[1].plot(K_range, silhouettes, 'go-')
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Score Silhouette')

plt.tight_layout()
plt.show()

# Meilleur K
best_k = K_range[np.argmax(silhouettes)]
print(f"\nMeilleur K (silhouette): {best_k}")

In [ ]:
# K-Means avec K optimal
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print(f"\nDistribution des clusters:")
print(pd.Series(clusters).value_counts().sort_index())

In [ ]:
# Visualisation des clusters avec ACP
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.title('Clusters K-Means visualises avec ACP')
plt.show()

## 7. Classification (Churn)

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Distribution train: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Distribution test: {pd.Series(y_test).value_counts().to_dict()}")

In [ ]:
# SMOTE pour equilibrer les classes
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Avant SMOTE: {len(y_train)}")
print(f"Apres SMOTE: {len(y_train_balanced)}")
print(f"Distribution: {pd.Series(y_train_balanced).value_counts().to_dict()}")

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_balanced, y_train_balanced)

y_pred_rf = rf.predict(X_test)

print("Random Forest:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Matrice de confusion
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de Confusion - Random Forest')
plt.ylabel('Vrai')
plt.xlabel('Predit')
plt.show()

In [ ]:
# Feature Importance
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(12, 8))
plt.title('Top 15 Features Importance')
plt.barh(range(15), importances[indices][::-1])
plt.yticks(range(15), [X.columns[i] for i in indices][::-1])
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Courbe ROC
from sklearn.metrics import roc_curve, roc_auc_score

y_prob_rf = rf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
auc = roc_auc_score(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', label=f'Random Forest (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC')
plt.legend()
plt.show()

## 8. Conclusions et Recommandations

### Resultats cles:

1. **Exploration des donnees**:
   - Dataset de 4373 clients avec 52 features
   - Valeurs manquantes dans Age (~30%), SupportTickets et Satisfaction (~5-8%)
   - Desequilibre de classes dans Churn

2. **ACP**:
   - Reduction de 52 features vers ~N composantes pour 95% de variance
   - Permet la visualisation 2D des clusters

3. **Clustering**:
   - 4 segments de clients identifies
   - Permet une segmentation marketing ciblee

4. **Classification**:
   - Random Forest avec SMOTE donne les meilleurs resultats
   - Features importantes: Recency, MonetaryTotal, ChurnRiskCategory

### Recommandations business:

1. **Clients a risque** (Churn predit = 1):
   - Programme de fidelisation personnalise
   - Offres speciales et reductions
   - Contact proactif du service client

2. **Segmentation**:
   - Champions: Programme VIP, avant-premieres
   - Fideles: Recompenses de fidelite
   - Potentiels: Incentives pour augmenter les achats
   - Dormants: Campagnes de reactivation